# 🤖 Autonomous Research Agent (ReAct-Based)
### Built with: Phi-2 + LangChain + DuckDuckGo + Wikipedia

---

**Tools Available:**  
- 🌐 DuckDuckGo (Web Search)  
- 📖 Wikipedia (Knowledge Base)  
- 🧠 Phi-2 (Reasoning + Report Generation)

---
**Agent Workflow (ReAct Loop):**  
`User Input` → `Thought` → `Action (Tool)` → `Observation` → `Execution Pipeline` → `Generate Report`
---
**Execution Pipeline:**  
`Thought` → `Action` → `Observation` → `Repeat` → `Final Report`
---


# Installing Dependencies

In [2]:
!pip install -q --upgrade \
    langchain \
    langchain-community \
    langchain-huggingface \
    transformers \
    accelerate \
    ddgs \
    wikipedia \
    huggingface_hub

print("✅ Dependencies installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 whic

# Import Libraries


In [3]:
# ============================================================
# CELL 2: Imports
# ============================================================

import torch
import wikipedia
import textwrap
from datetime import datetime

# Search
from ddgs import DDGS

# Hugging Face
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, pipeline

# LangChain wrapper
from langchain_huggingface import HuggingFacePipeline

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Running on: {device.upper()}")

🖥️ Running on: CUDA


## Load Phi-2 Model

In [4]:
# ============================================================
# CELL 3: Load Phi-2 Model
# ============================================================

MODEL_NAME = "microsoft/phi-2"

print("⚙️ Loading config...")
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.pad_token_id = 50256

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = 50256

print("⏳ Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("⚙️ Building pipeline...")
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.2,
    return_full_text=False,
    pad_token_id=50256,
    truncation=True
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("✅ Phi-2 ready")

⚙️ Loading config...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

⏳ Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


⚙️ Building pipeline...
✅ Phi-2 ready


# DuckDuckGo Search Tool

In [5]:
# ============================================================
# CELL 4: DuckDuckGo Search
# ============================================================

def search_web(topic: str, max_results: int = 3) -> str:
    print(f"🔍 Searching: {topic}")
    results = []

    try:
        with DDGS() as ddgs:
            search_results = list(ddgs.text(
                topic,
                max_results=max_results,
                region="wt-wt",
                safesearch="moderate"
            ))

            for i, r in enumerate(search_results, 1):
                title = r.get("title", "")
                body = r.get("body", "")
                results.append(f"[{i}] {title}:\n{body}")

        return "\n\n".join(results) if results else "No web results"

    except Exception as e:
        return f"Web search failed: {e}"

# Wikipedia Tool

In [6]:
# ============================================================
# CELL 5: Wikipedia
# ============================================================

def search_wikipedia(topic: str, sentences: int = 5) -> str:
    try:
        wikipedia.set_lang("en")
        topic = topic.replace(";", "").strip()

        results = wikipedia.search(topic)
        if not results:
            return "No Wikipedia content"

        title = results[0]

        try:
            return wikipedia.summary(title, sentences=sentences, auto_suggest=False)
        except wikipedia.exceptions.DisambiguationError as e:
            return wikipedia.summary(e.options[0], sentences=sentences)

    except Exception as e:
        return f"Wikipedia failed: {e}"

#Text Cleaning Utility

In [7]:
# ============================================================
# CELL 6: Utilities
# ============================================================

def format_text_block(text: str, width: int = 65) -> str:
    return "\n".join([
        textwrap.fill(p, width=width, break_long_words=False)
        if p.strip() else ""
        for p in text.split("\n")
    ])

def clean_and_combine(topic, web, wiki, max_chars=1500):
    combined = f"{wiki}\n\n{web}"
    return combined[:max_chars]

def create_cover_page(topic):
    return f"""
{'='*65}
{'AUTONOMOUS RESEARCH AGENT':^65}
{'Research Report':^65}
{'='*65}

Topic: {topic}
Date : {datetime.now().strftime("%B %d, %Y")}

{'='*65}
"""

#Generate Structured Report


In [8]:
# ============================================================
# CELL 7 (DETAILED): Section-by-Section Report Generator
# Strategy: Call Phi-2 once per section → better quality
# ============================================================


REPORT_SECTIONS = [
    {
        "id"         : "introduction",
        "title"      : "INTRODUCTION",
        "instruction": (
            "Write exactly 3 sentences about '{topic}'. "
            "Sentence 1: Define it clearly. "
            "Sentence 2: State its real-world importance. "
            "Sentence 3: State what this report will cover. "
            "Output only these 3 sentences. Nothing else."
        )
    },
    {
        "id"         : "key_findings",
        "title"      : "KEY FINDINGS",
        "instruction": (
            "Output exactly 5 bullet points about '{topic}'. "
            "Format: each line must start with • symbol. "
            "Each bullet = one specific fact or stat. "
            "Example format:\n"
            "• Solar costs dropped 89% in the last decade.\n"
            "• Wind capacity tripled globally since 2010.\n"
            "Follow this exact format. No intro sentence. "
            "No conclusion. Just 5 bullet points."
        )
    },
    {
        "id"         : "challenges",
        "title"      : "CHALLENGES",
        "instruction": (
            "Output exactly 4 bullet points about challenges in '{topic}'. "
            "Format: each line must start with • symbol. "
            "Cover one each: Technical, Ethical, Economic, Social. "
            "Example format:\n"
            "• Technical: [challenge description here].\n"
            "• Ethical: [challenge description here].\n"
            "Follow this exact format. No intro. Just 4 bullets."
        )
    },
    {
        "id"         : "future_scope",
        "title"      : "FUTURE SCOPE",
        "instruction": (
            "Write exactly 3 sentences about the future of '{topic}'. "
            "Sentence 1: One near-term trend (next 2-3 years). "
            "Sentence 2: One long-term possibility (10+ years). "
            "Sentence 3: Overall optimistic outlook. "
            "Output only 3 sentences. No bullet points. Nothing else."
        )
    },
    {
        "id"         : "conclusion",
        "title"      : "CONCLUSION",
        "instruction": (
            "Write exactly 3 sentences concluding '{topic}'. "
            "Sentence 1: Restate why the topic matters. "
            "Sentence 2: Name one key challenge to overcome. "
            "Sentence 3: One hopeful closing statement about the future. "
            "No citations. No bullet points. Exactly 3 sentences only."
        )
    }
]

def generate_section(
    topic      : str,
    instruction: str,
    context    : str,
    llm,
    section_num: int
) -> str:

    instruction = instruction.replace("{topic}", topic)

    prompt = f"""Instruct: You are a research writer. Be concise and direct.

Topic: {topic}
Task: {instruction}

Context: {context[:600]}

Write ONLY the section content. No titles. No preamble. Start immediately.

Response:"""

    try:
        response = llm.invoke(prompt)
        response = str(response).strip()

        # ── Artifact Removal ──────────────────────────────
        artifacts = [
            "Instruct:", "Response:", "Task:", "Context:",
            "Assistant:", "Output:", "```", "Note:",
            "Here are", "Here is", "The following",
            "As a research writer", "As an expert",
            "Write ONLY", "Section content:"
        ]
        for a in artifacts:
            response = response.replace(a, "")

        # ── Remove citation markers like (1), [2], [3] ───
        import re
        response = re.sub(r'\(\d+\)', '', response)
        response = re.sub(r'\[\d+\]', '', response)

        # ── Remove trailing symbols ────────────────────────
        response = response.strip(' "\'\n')

        # ── Quality check → fallback if too short ─────────
        if len(response.strip()) < 60:
            return generate_fallback(topic, section_num)

        return response.strip()

    except Exception as e:
        print(f"    ⚠️  Section {section_num} failed: {e}")
        return generate_fallback(topic, section_num)


def generate_fallback(topic: str, section_num: int) -> str:
    """
    Smart fallback content if Phi-2 fails for a section.
    """
    fallbacks = {
        1: (f"{topic} is a transformative field with growing global "
            f"importance. It has evolved significantly over recent years, "
            f"impacting industries ranging from healthcare to finance. "
            f"Understanding {topic} is essential for navigating the modern "
            f"technological landscape."),

        2: (f"• {topic} has seen rapid adoption across multiple industries.\n"
            f"• Research shows measurable improvements in efficiency and "
            f"accuracy compared to traditional methods.\n"
            f"• Real-world deployments have demonstrated significant "
            f"economic and social value.\n"
            f"• Leading organizations are actively investing in "
            f"{topic}-based solutions.\n"
            f"• Recent benchmarks show continued performance improvements "
            f"year over year."),

        3: (f"• High implementation costs remain a barrier for smaller "
            f"organizations.\n"
            f"• Data availability and quality significantly affect outcomes "
            f"in {topic}.\n"
            f"• Ethical and regulatory concerns require ongoing attention.\n"
            f"• Skilled talent shortages slow adoption and innovation.\n"
            f"• Interoperability with existing systems presents "
            f"technical challenges."),

        4: (f"The future of {topic} is marked by rapid innovation and "
            f"expanding applications. Emerging research directions promise "
            f"to overcome current limitations. Integration with "
            f"complementary technologies will unlock new possibilities. "
            f"Increased accessibility will drive broader adoption globally. "
            f"The next decade is expected to bring transformative "
            f"breakthroughs in this domain."),

        5: (f"{topic} represents one of the most significant developments "
            f"of our time. While challenges remain, the trajectory is "
            f"clearly positive. With continued research, collaboration, "
            f"and responsible development, {topic} has the potential to "
            f"create lasting benefits for society.")
    }
    return fallbacks.get(section_num, f"Content about {topic}.")


def format_section(number: int, title: str, content: str) -> str:

    WIDTH       = 65
    divider     = "─" * WIDTH
    thick_line  = "═" * WIDTH

    # ── Step 1: Split bullets that were merged on one line
    # Handles "• text1 • text2" → separate bullet lines
    import re
    content = re.sub(r'\s*•\s*', '\n• ', content)
    content = re.sub(r'\s*-\s{1,3}(?=[A-Z])', '\n• ', content)

    # ── Step 2: Process line by line ──────────────────────
    lines = content.split('\n')
    formatted_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if line.startswith('•'):
            # Bullet point — wrap with hanging indent
            wrapped = textwrap.fill(
                line,
                width             = WIDTH,
                initial_indent    = '  ',
                subsequent_indent = '    ',
                break_long_words  = False
            )
            formatted_lines.append(wrapped)

        else:
            # Paragraph text — wrap normally
            wrapped = textwrap.fill(
                line,
                width            = WIDTH,
                break_long_words = False
            )
            if wrapped:
                formatted_lines.append(wrapped)

    # ── Step 3: Enforce max line count per section ────────
    # Prevents Phi-2 from generating excessively long sections
    MAX_LINES = {
        1: 8,   # Introduction  — max 8 lines
        2: 10,  # Key Findings  — max 10 lines (bullets)
        3: 12,  # Challenges    — max 12 lines (bullets)
        4: 8,   # Future Scope  — max 8 lines
        5: 6    # Conclusion    — max 6 lines
    }

    limit = MAX_LINES.get(number, 10)
    if len(formatted_lines) > limit:
        formatted_lines = formatted_lines[:limit]
        formatted_lines.append("  [...]")

    body = "\n".join(formatted_lines)

    return (
        f"\n{thick_line}\n"
        f"  {number}. {title}\n"
        f"{divider}\n"
        f"{body}\n"
    )


def generate_detailed_report(
    topic  : str,
    context: str,
    llm
) -> str:
    """
    Main function: generates all 5 sections one by one.
    Each section gets its own focused Phi-2 call.

    Args:
        topic   : Research topic
        context : Combined web + wiki content
        llm     : HuggingFacePipeline

    Returns:
        Complete formatted report body
    """

    print(f"\n📝 Generating detailed report in {len(REPORT_SECTIONS)} sections...")
    print("   (Each section is generated separately for quality)\n")

    all_sections = []

    for i, section in enumerate(REPORT_SECTIONS, 1):

        print(f"  ✍️  [{i}/{len(REPORT_SECTIONS)}] "
              f"Writing: {section['title']}...")

        # Generate this section
        content = generate_section(
            topic       = topic,
            instruction = section["instruction"],
            context     = context,
            llm         = llm,
            section_num = i
        )

        # Format and add to report
        formatted = format_section(i, section["title"], content)
        all_sections.append(formatted)

        print(f"             ✅ Done ({len(content)} chars)")

    print(f"\n✅ All {len(REPORT_SECTIONS)} sections generated!")
    return "\n".join(all_sections)

#Save Report Utility


In [9]:
# ============================================================
# CELL 8: Save
# ============================================================

def save_report(topic, text):
    filename = topic.replace(" ", "_") + ".txt"
    with open(filename, "w") as f:
        f.write(format_text_block(text))
    print("Saved:", filename)

In [10]:
# ============================================================
# CELL 9 (FIXED): Rule-Based ReAct Agent
# ============================================================

TOOLS = {
    "search_web"        : search_web,
    "search_wikipedia"  : search_wikipedia,
    "generate_report"   : generate_detailed_report
}

def agent_decide(steps_done: list) -> str:
    """
    Rule-based agent decision function.
    Follows a logical ReAct sequence every time.

    Thought process:
      - No web data yet?      → search_web
      - No wiki data yet?     → search_wikipedia
      - Have both sources?    → generate_report
      - Report done?          → done
    """
    if "search_web" not in steps_done:
        thought = "I don't have web data yet. I should search the web."
        action  = "search_web"

    elif "search_wikipedia" not in steps_done:
        thought = "I have web data but need encyclopedic context."
        action  = "search_wikipedia"

    elif "generate_report" not in steps_done:
        thought = "I have all context needed. Time to write the report."
        action  = "generate_report"

    else:
        thought = "Report is complete. My work is done."
        action  = "done"

    return thought, action


def run_react_agent():
    """
    ReAct Agent Loop — Rule-Based Decision + Tool Execution
    Pattern: Thought → Action → Observe → Repeat
    """

    print("=" * 65)
    print("     🤖 REACT RESEARCH AGENT — STARTING")
    print("=" * 65)

    # ── Get Topic ─────────────────────────────────────────
    topic = input("\n📌 Enter research topic: ").strip()
    if not topic:
        print("❌ No topic entered.")
        return

    print(f"\n✅ Topic: '{topic}'")
    print("-" * 65)

    # ── Agent Memory (State) ──────────────────────────────
    memory = {
        "topic"      : topic,
        "web"        : "",
        "wiki"       : "",
        "context"    : "",
        "report"     : "",
        "steps_done" : []
    }

    max_steps  = 6
    step_count = 0
    done       = False

    # ── ReAct Loop ────────────────────────────────────────
    while not done and step_count < max_steps:

        step_count += 1
        print(f"\n{'─'*65}")
        print(f"  🔁 STEP {step_count}")

        # ── THOUGHT ───────────────────────────────────────
        thought, action = agent_decide(memory["steps_done"])
        print(f"  💭 THOUGHT : {thought}")
        print(f"  ⚡ ACTION  : {action}")

        # ── ACTION + OBSERVE ──────────────────────────────
        if action == "search_web":
            print(f"  👁  OBSERVE : Running web search...")
            result         = search_web(topic, max_results=3)
            memory["web"]  = result
            memory["steps_done"].append("search_web")
            memory["context"] = clean_and_combine(
                topic, memory["web"], memory["wiki"]
            )
            print(f"  ✅ Web search complete "
                  f"({len(result)} chars)")

        elif action == "search_wikipedia":
            print(f"  👁  OBSERVE : Running Wikipedia search...")
            result          = search_wikipedia(topic, sentences=5)
            memory["wiki"]  = result
            memory["steps_done"].append("search_wikipedia")
            memory["context"] = clean_and_combine(
                topic, memory["web"], memory["wiki"]
            )
            print(f"  ✅ Wikipedia complete "
                  f"({len(result)} chars)")

        elif action == "generate_report":
            print(f"  👁  OBSERVE : Generating detailed report...")
            memory["report"] = generate_detailed_report(
                topic, memory["context"], llm
            )
            memory["steps_done"].append("generate_report")
            print(f"  ✅ Report generated.")
            done = True

        elif action == "done":
            print("  🏁 Agent decided it is done.")
            done = True

    # ── Build Final Report ─────────────────────────────────
    cover = create_cover_page(topic)

    agent_trace = (
        f"\n{'='*65}\n"
        f"{'AGENT EXECUTION TRACE':^65}\n"
        f"{'='*65}\n"
        f"  Steps : {' → '.join(memory['steps_done'])}\n"
        f"  Total : {step_count} step(s)\n"
    )

    footer = (
        f"\n{'='*65}\n"
        f"{'END OF REPORT':^65}\n"
        f"{'='*65}\n"
        f"  Generated by : ReAct Research Agent\n"
        f"  Model        : Microsoft Phi-2\n"
        f"  Tools        : DuckDuckGo + Wikipedia\n"
    )

    full_report = (
        cover
        + agent_trace
        + "\n" + "="*65 + "\n"
        + memory["report"]
        + footer
    )

    print("\n" + "="*65)
    print("          📋 FINAL REPORT")
    print("="*65)
    print(full_report)

    save_report(topic, full_report)
    print(f"\n✅ Agent complete! Report saved.")
    return full_report

# **CELL 10: RUN**

In [12]:
report = run_react_agent()

     🤖 REACT RESEARCH AGENT — STARTING

📌 Enter research topic: Artificial Intelligence in Modern Defence Systems

✅ Topic: 'Artificial Intelligence in Modern Defence Systems'
-----------------------------------------------------------------

─────────────────────────────────────────────────────────────────
  🔁 STEP 1
  💭 THOUGHT : I don't have web data yet. I should search the web.
  ⚡ ACTION  : search_web
  👁  OBSERVE : Running web search...
🔍 Searching: Artificial Intelligence in Modern Defence Systems
  ✅ Web search complete (940 chars)

─────────────────────────────────────────────────────────────────
  🔁 STEP 2
  💭 THOUGHT : I have web data but need encyclopedic context.
  ⚡ ACTION  : search_wikipedia
  👁  OBSERVE : Running Wikipedia search...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Wikipedia complete (853 chars)

─────────────────────────────────────────────────────────────────
  🔁 STEP 3
  💭 THOUGHT : I have all context needed. Time to write the report.
  ⚡ ACTION  : generate_report
  👁  OBSERVE : Generating detailed report...

📝 Generating detailed report in 5 sections...
   (Each section is generated separately for quality)

  ✍️  [1/5] Writing: INTRODUCTION...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             ✅ Done (563 chars)
  ✍️  [2/5] Writing: KEY FINDINGS...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             ✅ Done (748 chars)
  ✍️  [3/5] Writing: CHALLENGES...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             ✅ Done (1018 chars)
  ✍️  [4/5] Writing: FUTURE SCOPE...


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             ✅ Done (885 chars)
  ✍️  [5/5] Writing: CONCLUSION...
             ✅ Done (889 chars)

✅ All 5 sections generated!
  ✅ Report generated.

          📋 FINAL REPORT

                    AUTONOMOUS RESEARCH AGENT                    
                         Research Report                         

Topic: Artificial Intelligence in Modern Defence Systems
Date : March 26, 2026


                      AGENT EXECUTION TRACE                      
  Steps : search_web → search_wikipedia → generate_report
  Total : 3 step(s)


═════════════════════════════════════════════════════════════════
  1. INTRODUCTION
─────────────────────────────────────────────────────────────────
An AI arms race has been observed since 2010 among multiple
nations as they strive to acquire superior AI capabilities within
their defence sectors. This report aims to provide insights into
various aspects related to modern warfare's integration with
cutting-edge artificial intelligence technologies, including
